# Section 5.8 — Sensitivity to Horizon Length

Reproduces the three rolling-horizon experiments from thesis Section 5.8.

**Sub-experiments**

| # | Description | Instance | Variants | Real horizon |
|---|-------------|----------|----------|-------------|
| A | Deterministic (EV-MILP) rolling horizon | 60 cages | DET-30M, DET-60M | 120 months |
| B | Stochastic SP rolling horizon (small fleet) | 30 cages | 30M (9 scen), 30M_81 (81 scen), 60M (81 scen) | 120 months |
| C | Stochastic SP with deterministic tail (large fleet) | 60 cages | 30M (27 scen), 60M+det-tail (27 scen) | 60 months |

**Runtime warning**: Parts B and C are computationally expensive (hours to days).
Results are saved incrementally to `runs/` so cells can be safely re-run.

**Dependencies** (all local to this folder):
- `ip.py` — per-scenario MILP subproblem
- `sp_rh.py` — tree-aware B-PHA solver (parameterised horizon/stages)
- `sp_60m.py` — 60-month SP with deterministic extension tail
- `extract_scenario.py` — per-scenario decision extraction from solved SP
- `forward_sim_long.py` — forward simulator for multi-month commit windows
- `instance.py` — 30-cage fleet (used directly by `sp_rh`; 60-cage data defined inline)


In [6]:
import os, sys, time, pickle, json
import numpy as np
import pandas as pd

# ── Path setup ──────────────────────────────────────────────────────────────
# models/ provides the full SalmonFarmingMILP (used by ip.py via models/IP.py path)
# Local directory provides sp_rh, sp_60m, extract_scenario, forward_sim_long, instance
HERE = os.path.dirname(os.path.abspath('__file__'))
MODELS = os.path.normpath(os.path.join(HERE, '..', 'models'))
for p in [MODELS, HERE]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Local imports (all live in this folder after setup)
from scripts.forward_sim_long import advance_state_long
from scripts.extract_scenario import (
    find_scenario_idx_by_labels,
    find_best_match_scenario,
    extract_scenario_decisions,
    feasibility_weighted_expected_obj,
    scenario_slack_summary,
)
import scripts.sp_rh as sp_rh
import scripts.sp_60m as sp_60m

print("Imports OK")
print(f"  HERE   = {HERE}")
print(f"  MODELS = {MODELS}")


Imports OK
  HERE   = c:\Users\Isak\OneDrive - University of Bergen\Dokumenter\Programmeringsfiler\master\fork_ulrik_models\experimental_evaluation\5.8_horizon_experiment
  MODELS = c:\Users\Isak\OneDrive - University of Bergen\Dokumenter\Programmeringsfiler\master\fork_ulrik_models\experimental_evaluation\models


In [2]:
# ── Economic constants (matching IP.py) ────────────────────────────────────
ANNUAL_RATE            = 0.10
SMOLT_COST_PER_HEAD    = 10.0        # NOK
TERMINAL_VALUE_PER_KG  = 60.0        # NOK/kg
FEED_COST_PER_KG_MONTH = 1.0         # NOK/kg/month  (tracked inside MILP)

PRICE_BREAKS = [                     # (lo_kg, hi_kg, NOK/kg)
    (1.0, 2.0, 39.72), (2.0, 3.0, 52.66), (3.0, 4.0, 60.73),
    (4.0, 5.0, 63.33), (5.0, 6.0, 64.55), (6.0, 7.0, 64.14),
    (7.0, 8.0, 62.85), (8.0, 9.0, 61.32), (9.0, 1e9,  58.80),
]

def price_for_weight_g(weight_g):
    w = weight_g / 1000.0
    for lo, hi, p in PRICE_BREAKS:
        if lo <= w < hi:
            return p
    return 58.80

def discount_factor(t_months, rate=ANNUAL_RATE):
    return (1 + rate) ** (-t_months / 12.0)


def compute_npv(real_state_log, units_df_final, real_horizon_months):
    """
    Compute discounted NPV from a forward-simulation log.
    Includes: harvest revenue, smolt purchase cost, terminal value.
    Feed costs are implicitly captured in the MILP objective but not
    separately tracked here (they partially cancel across variants).
    """
    npv = 0.0

    for entry in real_state_log:
        t_start = int(entry.get('t_start', 0))
        fwd_log = entry.get('fwd_log', [])

        for ev in fwd_log:
            t   = t_start + int(ev.get('month', 0))
            d   = discount_factor(t)
            evt = ev.get('event', '')

            if evt in ('harvest_existing', 'harvest_new'):
                count  = float(ev.get('count_before', 0))
                wg     = float(ev.get('weight_g_before', 0))
                bio_kg = count * wg / 1000.0
                npv   += d * bio_kg * price_for_weight_g(wg)

            elif evt == 'stocked_new':
                q    = float(ev.get('q', 0))
                npv -= d * q * SMOLT_COST_PER_HEAD

    # Terminal value: remaining standing biomass at horizon end
    if units_df_final is not None:
        for _, row in units_df_final.iterrows():
            cnt = row.get('count');  wg = row.get('avg_weight_g')
            if cnt is not None and wg is not None:
                if not (pd.isna(cnt) or pd.isna(wg)):
                    bio_kg = float(cnt) * float(wg) / 1000.0
                    npv   += discount_factor(real_horizon_months) * TERMINAL_VALUE_PER_KG * bio_kg

    return npv


def load_result(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

print("Helpers defined.")


Helpers defined.


## Part A — Deterministic Horizon Comparison (DET-30M vs DET-60M)

Rolls the deterministic MILP (EV-style, all-normal temperatures) across a
120-month real horizon under eight named temperature paths.

| Setting | Value |
|---------|-------|
| Planning horizons | 30 months, 60 months |
| Commit window | 12 months per roll |
| Rolls per path | 10 |
| Temperature paths | 4 archetypes + 4 random (seed=0) |
| MIP gap | 3% |
| Instance | 60-cage regional fleet (6 locations) |

The planner always assumes normal temperatures inside the MILP;
only the realised temperatures fed to the forward simulator vary.


In [3]:
# ── 60-cage instance (rolling_horizon_experiment fleet) ───────────────────
# Loc 1–6 active; matching the instance used in the original experiment.
import numpy as np, pandas as pd

_60c_rows = [
    # Loc1
    {'unit':'Unit 1',  'count':150_000.0,'avg_weight_g':4500.0,'volume_m3':35_000,'location':'Loc1'},
    {'unit':'Unit 2',  'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':35_000,'location':'Loc1'},
    {'unit':'Unit 3',  'count':155_000.0,'avg_weight_g':5000.0,'volume_m3':35_000,'location':'Loc1'},
    {'unit':'Unit 4',  'count':156_500.0,'avg_weight_g':5300.0,'volume_m3':35_000,'location':'Loc1'},
    {'unit':'Unit 5',  'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':35_000,'location':'Loc1'},
    {'unit':'Unit 6',  'count':150_000.0,'avg_weight_g':4500.0,'volume_m3':35_000,'location':'Loc1'},
    {'unit':'Unit 7',  'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':35_000,'location':'Loc1'},
    {'unit':'Unit 8',  'count':187_000.0,'avg_weight_g':550.0,'volume_m3':35_000,'location':'Loc1'},
    {'unit':'Unit 9',  'count':190_000.0,'avg_weight_g':500.0,'volume_m3':35_000,'location':'Loc1'},
    {'unit':'Unit 10', 'count':190_000.0,'avg_weight_g':500.0,'volume_m3':35_000,'location':'Loc1'},
    # Loc2
    {'unit':'Unit 1',  'count':152_000.0,'avg_weight_g':4800.0,'volume_m3':35_000,'location':'Loc2'},
    {'unit':'Unit 2',  'count':160_000.0,'avg_weight_g':5100.0,'volume_m3':35_000,'location':'Loc2'},
    {'unit':'Unit 3',  'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':35_000,'location':'Loc2'},
    {'unit':'Unit 4',  'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':35_000,'location':'Loc2'},
    {'unit':'Unit 5',  'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':35_000,'location':'Loc2'},
    {'unit':'Unit 6',  'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':35_000,'location':'Loc2'},
    {'unit':'Unit 7',  'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':35_000,'location':'Loc2'},
    {'unit':'Unit 8',  'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':35_000,'location':'Loc2'},
    {'unit':'Unit 9',  'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':35_000,'location':'Loc2'},
    {'unit':'Unit 10', 'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':35_000,'location':'Loc2'},
    {'unit':'Unit 11', 'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':35_000,'location':'Loc2'},
    {'unit':'Unit 12', 'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':35_000,'location':'Loc2'},
    # Loc3
    {'unit':'Unit 1',  'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':30_000,'location':'Loc3'},
    {'unit':'Unit 2',  'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':30_000,'location':'Loc3'},
    {'unit':'Unit 3',  'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':30_000,'location':'Loc3'},
    {'unit':'Unit 4',  'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':30_000,'location':'Loc3'},
    {'unit':'Unit 5',  'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':30_000,'location':'Loc3'},
    {'unit':'Unit 6',  'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':30_000,'location':'Loc3'},
    {'unit':'Unit 7',  'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':30_000,'location':'Loc3'},
    {'unit':'Unit 8',  'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':30_000,'location':'Loc3'},
    # Loc4
    {'unit':'Unit 1',  'count':145_000.0,'avg_weight_g':5700.0,'volume_m3':35_000,'location':'Loc4'},
    {'unit':'Unit 2',  'count':155_000.0,'avg_weight_g':5000.0,'volume_m3':35_000,'location':'Loc4'},
    {'unit':'Unit 3',  'count':156_500.0,'avg_weight_g':5300.0,'volume_m3':35_000,'location':'Loc4'},
    {'unit':'Unit 4',  'count':150_000.0,'avg_weight_g':4500.0,'volume_m3':35_000,'location':'Loc4'},
    {'unit':'Unit 5',  'count':184_000.0,'avg_weight_g':890.0,'volume_m3':35_000,'location':'Loc4'},
    {'unit':'Unit 6',  'count':173_000.0,'avg_weight_g':1250.0,'volume_m3':35_000,'location':'Loc4'},
    {'unit':'Unit 7',  'count':190_000.0,'avg_weight_g':500.0,'volume_m3':35_000,'location':'Loc4'},
    {'unit':'Unit 8',  'count':190_000.0,'avg_weight_g':500.0,'volume_m3':35_000,'location':'Loc4'},
    {'unit':'Unit 9',  'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':35_000,'location':'Loc4'},
    {'unit':'Unit 10', 'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':35_000,'location':'Loc4'},
    # Loc5
    {'unit':'Unit 1',  'count':148_000.0,'avg_weight_g':5500.0,'volume_m3':30_000,'location':'Loc5'},
    {'unit':'Unit 2',  'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':30_000,'location':'Loc5'},
    {'unit':'Unit 3',  'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':30_000,'location':'Loc5'},
    {'unit':'Unit 4',  'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':30_000,'location':'Loc5'},
    {'unit':'Unit 5',  'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':30_000,'location':'Loc5'},
    {'unit':'Unit 6',  'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':30_000,'location':'Loc5'},
    {'unit':'Unit 7',  'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':30_000,'location':'Loc5'},
    {'unit':'Unit 8',  'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':30_000,'location':'Loc5'},
    # Loc6
    {'unit':'Unit 1',  'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':35_000,'location':'Loc6'},
    {'unit':'Unit 2',  'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':35_000,'location':'Loc6'},
    {'unit':'Unit 3',  'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':35_000,'location':'Loc6'},
    {'unit':'Unit 4',  'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':35_000,'location':'Loc6'},
    {'unit':'Unit 5',  'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':35_000,'location':'Loc6'},
    {'unit':'Unit 6',  'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':35_000,'location':'Loc6'},
    {'unit':'Unit 7',  'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':35_000,'location':'Loc6'},
    {'unit':'Unit 8',  'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':35_000,'location':'Loc6'},
    {'unit':'Unit 9',  'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':35_000,'location':'Loc6'},
    {'unit':'Unit 10', 'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':35_000,'location':'Loc6'},
    {'unit':'Unit 11', 'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':35_000,'location':'Loc6'},
    {'unit':'Unit 12', 'count':float('nan'),'avg_weight_g':float('nan'),'volume_m3':35_000,'location':'Loc6'},
]
units_df_60c = pd.DataFrame(_60c_rows)

loc_mab_60c = {
    'Location 1': 4_000_000, 'Location 2': 5_600_000, 'Location 3': 2_500_000,
    'Location 4': 4_300_000, 'Location 5': 2_600_000, 'Location 6': 5_500_000,
}
regional_mab_60c = 35_000_000

# Temperature profiles (seasonal, 12-month cycle)
temps_bad_12    = np.array([3,   3,   3,   4,   7,  10,  12,  14.5, 13.5, 11,   8,   5.5])
temps_normal_12 = np.array([5,   5,   5,   6,   9,  12,  14,  16.5, 15.5, 13,  10,   7.5])
temps_good_12   = np.array([7,   7,   7,   8,  11,  14,  16,  18.5, 17.5, 15,  12,   9.5])
temp_map_60c = {'bad': temps_bad_12, 'normal': temps_normal_12, 'good': temps_good_12}

S_normal_60c = (1.0 - 0.0002) ** 30
S_bad_60c    = (1.0 - 0.001)  ** 30

print(f"60-cage instance loaded: {len(units_df_60c)} rows across {units_df_60c['location'].nunique()} locations")


60-cage instance loaded: 60 rows across 6 locations


In [4]:
# ── Part A: Temperature paths and DET experiment helpers ───────────────────

DET_REAL_HORIZON = 120    # months
DET_COMMIT       = 12     # months per roll
DET_N_ROLLS      = DET_REAL_HORIZON // DET_COMMIT   # = 10
DET_BLOCK_MONTHS = 12     # one temperature label per block

LABELS = ('bad', 'normal', 'good')

DET_ARCHETYPES = [
    ('normal',  ['normal'] * 10),
    ('warm',    ['normal','good','normal','good','normal','good','normal','good','normal','good']),
    ('cold',    ['normal','bad','normal','bad','normal','bad','normal','bad','normal','bad']),
    ('mixed',   ['normal','good','bad','normal','good','bad','normal','good','bad','normal']),
]

def det_sample_paths(n_extra=4, seed=0):
    paths = [{'name': n, 'blocks': list(b)} for n, b in DET_ARCHETYPES]
    rng = np.random.default_rng(seed)
    for i in range(n_extra):
        blocks = [str(rng.choice(LABELS)) for _ in range(DET_N_ROLLS)]
        paths.append({'name': f'random_{i:02d}', 'blocks': blocks})
    return paths

def det_build_realised(path_blocks, start_cal, temp_map, S_normal, S_bad):
    """Build per-month realised temps and survival for 120 months."""
    monthly = [lbl for lbl in path_blocks for _ in range(DET_BLOCK_MONTHS)]
    temps = np.zeros(DET_REAL_HORIZON)
    S     = np.full(DET_REAL_HORIZON, S_normal)
    prev_block = None
    for i, lbl in enumerate(monthly):
        if i % DET_BLOCK_MONTHS == 0:
            S_block = S_bad if (lbl == 'good' and prev_block == 'good') else S_normal
            prev_block = lbl
        temps[i] = temp_map[lbl][(start_cal + i) % 12]
        S[i]     = S_block
    return temps, S

def det_extract_decisions(milp, n_impl):
    """Extract the first n_impl months of decisions from a solved MILP."""
    z, q, h_e, h = {}, {}, {}, {}
    z_vars = milp.variables.get('s', {}) or milp.variables.get('z', {})
    for (u, t), var in z_vars.items():
        if int(t) < n_impl: z[(u, int(t))] = int(round(float(var.X)))
    for (u, t), var in milp.variables.get('q', {}).items():
        if int(t) < n_impl: q[(u, int(t))] = float(var.X)
    for (u, t), var in milp.variables.get('h_exist', {}).items():
        if int(t) < n_impl: h_e[(u, int(t))] = int(round(float(var.X)))
    for (u, ss, t), var in milp.variables.get('h', {}).items():
        if int(t) < n_impl: h[(u, int(ss), int(t))] = int(round(float(var.X)))
    return {'z': z, 'q': q, 'h_exist': h_e, 'h': h}

print("Part A helpers ready.")


Part A helpers ready.


In [ ]:
# ── Part A: Run DET-30M and DET-60M ────────────────────────────────────────
# Runtime: ~10-30 min total (DET MILPs solve in a few seconds each).
# Results are saved incrementally; already-completed cells are skipped.

from scripts.ip import SalmonFarmingMILP as LocalMILP

DET_SAVE_DIR = os.path.join(HERE, 'runs', 'det')
DET_MIP_GAP  = 0.03
DET_HORIZONS = [30, 60]    # planning horizons to compare
DET_PATHS    = det_sample_paths(n_extra=4, seed=0)

os.makedirs(DET_SAVE_DIR, exist_ok=True)

def run_det_path(horizon, path_spec, mip_gap=DET_MIP_GAP, start_cal=0):
    """Run one (horizon, path) DET rolling-horizon combination."""
    out_dir  = os.path.join(DET_SAVE_DIR, f'det_{horizon}m', path_spec['name'])
    pkl_path = os.path.join(out_dir, 'result.pkl')
    if os.path.exists(pkl_path):
        existing = load_result(pkl_path)
        if not existing.get('config', {}).get('partial', True):
            print(f'  [SKIP] det_{horizon}m / {path_spec["name"]} — already done.')
            return existing

    os.makedirs(out_dir, exist_ok=True)
    real_temps, real_S = det_build_realised(
        path_spec['blocks'], start_cal, temp_map_60c, S_normal_60c, S_bad_60c
    )
    units_df = units_df_60c.copy()
    cal      = start_cal % 12
    rolls, real_state_log = [], []

    overall_t0 = time.time()
    for k in range(DET_N_ROLLS):
        t0 = k * DET_COMMIT;  t1 = min(t0 + DET_COMMIT, DET_REAL_HORIZON)

        # Build normal-temperature plan for this horizon
        n = int(horizon)
        t_normal = np.array([float(temp_map_60c['normal'][(cal + t) % 12]) for t in range(n)])
        S_plan   = np.full(n, S_normal_60c)

        t_s0 = time.time()
        milp = LocalMILP(
            units_df=units_df, temps_t=t_normal, survival_rates=S_plan,
            loc_mab=loc_mab_60c, regional_mab=regional_mab_60c,
            horizon_months=n, scenario_name='det_normal',
        )
        milp.model.Params.OutputFlag = 0
        milp.model.Params.MIPGap     = mip_gap
        milp.model.Params.TimeLimit  = 120.0
        milp.model.optimize()
        wall = time.time() - t_s0

        if milp.model.SolCount == 0:
            print(f'  [WARN] roll {k}: no solution for det_{horizon}m / {path_spec["name"]}')
            decisions = {'z': {}, 'q': {}, 'h_exist': {}, 'h': {}}
        else:
            decisions = det_extract_decisions(milp, DET_COMMIT)

        n_impl   = t1 - t0
        temps_w  = real_temps[t0:t1]
        S_w      = real_S[t0:t1]
        new_df, fwd_log = advance_state_long(units_df, decisions, temps_w, S_w)
        for ev in fwd_log: ev['t_real_offset'] = t0

        rolls.append({
            'roll_idx': k, 't_start': t0, 'horizon': horizon,
            'wallclock_s': wall, 'calendar_month': cal,
            'objval': float(milp.model.ObjVal) if milp.model.SolCount > 0 else float('nan'),
        })
        real_state_log.append({
            'solve_idx': k, 't_start': t0, 'n_implement': n_impl,
            'fwd_log': fwd_log, 'decisions': decisions,
            'realised_temps': list(map(float, temps_w)),
            'realised_S':     list(map(float, S_w)),
            'units_df_before': units_df.copy(),
            'units_df_after':  new_df.copy(),
        })
        units_df = new_df
        cal      = (cal + DET_COMMIT) % 12

    result = {
        'config': {
            'variant': f'DET-{horizon}M', 'horizon_months': horizon,
            'commit_months': DET_COMMIT, 'n_rolls': DET_N_ROLLS,
            'real_horizon_months': DET_REAL_HORIZON,
            'path_name': path_spec['name'], 'path_blocks': path_spec['blocks'],
            'mip_gap': mip_gap, 'partial': False,
            'total_wallclock_s': time.time() - overall_t0,
        },
        'rolls': rolls, 'real_state_log': real_state_log, 'units_df_final': units_df,
    }
    with open(pkl_path, 'wb') as f: pickle.dump(result, f)
    total_wall = sum(r['wallclock_s'] for r in rolls)
    print(f'  Done det_{horizon}m / {path_spec["name"]}  '
          f'solve_wall={total_wall:.1f}s  → {pkl_path}')
    return result

# Run all combinations
print(f'Running DET experiment: horizons={DET_HORIZONS}, paths={[p["name"] for p in DET_PATHS]}')
print(f'Saving to: {DET_SAVE_DIR}')
t0_all = time.time()

for horizon in DET_HORIZONS:
    for path_spec in DET_PATHS:
        run_det_path(horizon, path_spec)

print(f'\nPart A complete. Total wall: {time.time()-t0_all:.1f}s')


Running DET experiment: horizons=[30, 60], paths=['normal', 'warm', 'cold', 'mixed', 'random_00', 'random_01', 'random_02', 'random_03']
Saving to: c:\Users\Isak\OneDrive - University of Bergen\Dokumenter\Programmeringsfiler\master\fork_ulrik_models\experimental_evaluation\5.8_horizon_experiment\runs\det
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2718474
Academic license 2718474 - for non-commercial use only - registered to ie___@uib.no
  Done det_30m / normal  solve_wall=107.2s  → c:\Users\Isak\OneDrive - University of Bergen\Dokumenter\Programmeringsfiler\master\fork_ulrik_models\experimental_evaluation\5.8_horizon_experiment\runs\det\det_30m\normal\result.pkl


KeyboardInterrupt: 

In [ ]:
# ── Part A: NPV results table ──────────────────────────────────────────────

rows = []
for horizon in DET_HORIZONS:
    for path_spec in DET_PATHS:
        pkl = os.path.join(DET_SAVE_DIR, f'det_{horizon}m', path_spec['name'], 'result.pkl')
        if not os.path.exists(pkl):
            print(f'  Missing: det_{horizon}m / {path_spec["name"]} — run the cell above first.')
            continue
        res = load_result(pkl)
        npv = compute_npv(res['real_state_log'], res['units_df_final'], DET_REAL_HORIZON)
        wall = sum(r['wallclock_s'] for r in res['rolls'])
        rows.append({
            'Variant':    f'DET-{horizon}M',
            'Path':       path_spec['name'],
            'NPV [BNOK]': round(npv / 1e9, 3),
            'Wall [s]':   round(wall, 1),
        })

if rows:
    df_det = pd.DataFrame(rows).pivot(index='Path', columns='Variant', values='NPV [BNOK]')
    print('Realised NPV by (path, variant) [BNOK] — excludes feed costs\n')
    display(df_det)

    # Mean NPV per variant
    mean_npv = pd.DataFrame(rows).groupby('Variant')['NPV [BNOK]'].mean()
    print('\nMean NPV across paths [BNOK]:')
    display(mean_npv.to_frame())


## Part B — Stochastic Horizon Comparison, 30-cage fleet

Compares three SP variants rolled across 120 months on the 30-cage
(3-location) fleet. The instance is loaded from `instance.py` (30 cages).

| Variant | Horizon | Stages | Scenarios | Commit | Solves |
|---------|---------|--------|-----------|--------|--------|
| `30M`    | 30 mo  | 2      | 9 (3²)    | 15 mo  | 8      |
| `30M_81` | 30 mo  | 4      | 81 (3⁴)   | 15 mo  | 8      |
| `60M`    | 60 mo  | 4      | 81 (3⁴)   | 45 mo  | 3      |

Five temperature trajectories: `normal`, `warm`, `cold`, `oscillating`, `stress_then_recover`.

**Runtime**: each (variant × trajectory) cell takes ~30 min (`30M`) to many hours
(`60M`). The 30-cage `60M` variant is the most expensive; start with `30M` to
verify the pipeline. Results are saved incrementally; completed cells are skipped.


In [ ]:
# ── Part B: 30-cage SP experiment configuration ────────────────────────────
# instance.py (in this folder) provides the 30-cage fleet used by sp_rh.py.
from scripts.instance import (
    units_df as units_df_30c,
    loc_mab  as loc_mab_30c,
    regional_mab as regional_mab_30c,
    temps_bad_12  as tB12,
    temps_normal_12 as tN12,
    temps_good_12   as tG12,
    S_normal_v as S_normal_30c,
    S_bad_v    as S_bad_30c,
)
temp_map_30c = {'bad': tB12, 'normal': tN12, 'good': tG12}

STAGE_LEN = 15    # months per stage
REAL_HORIZON_SP = 120

VARIANT_CFG = {
    '30M':    {'T': 30,  'commit': 15, 'n_stages': 2, 'stage_slices': None,
               'macro_block_len': 1},
    '30M_81': {'T': 30,  'commit': 15, 'n_stages': 4,
               'stage_slices': [list(range(0, 7)), list(range(7, 15)),
                                 list(range(15, 22)), list(range(22, 30))],
               'macro_block_len': 2},
    '60M':    {'T': 60,  'commit': 45, 'n_stages': 4, 'stage_slices': None,
               'macro_block_len': 1},
}

TRAJECTORY_BLOCKS = {
    'normal':              ['normal'] * 10,
    'warm':                ['good']   * 10,
    'cold':                ['bad']    * 10,
    'oscillating':         ['good', 'bad'] * 5,
    'stress_then_recover': ['bad'] * 4 + ['normal'] * 6,
}

def make_stage_slices(n_stages, stage_len=STAGE_LEN):
    return [list(range(k * stage_len, (k + 1) * stage_len)) for k in range(n_stages)]

def build_realized_sp(blocks, n_months, temp_map, S_normal, S_bad, start_cal=0):
    """Build per-month realised temps + survival for n_months, 15-month blocks."""
    temps = np.zeros(n_months)
    S     = np.zeros(n_months)
    for t in range(n_months):
        bi   = t // STAGE_LEN
        lbl  = blocks[bi] if bi < len(blocks) else 'normal'
        prev = blocks[bi-1] if bi > 0 else None
        S[t] = S_bad if (lbl == 'good' and prev == 'good') else S_normal
        temps[t] = temp_map[lbl][(start_cal + t) % 12]
    return temps, S

def derive_stage_labels(blocks, t_start, n_stages, stage_offsets=None):
    offsets = stage_offsets or [k * STAGE_LEN for k in range(n_stages)]
    labels = []
    for off in offsets:
        bi  = (t_start + off) // STAGE_LEN
        lbl = blocks[bi] if bi < len(blocks) else 'normal'
        labels.append(lbl)
    return tuple(labels)

print(f'30-cage instance: {len(units_df_30c)} cage rows across {units_df_30c["location"].nunique()} locations')
print(f'Variants: {list(VARIANT_CFG)}')
print(f'Trajectories: {list(TRAJECTORY_BLOCKS)}')


In [ ]:
# ── Part B: Run SP 30-cage rolling-horizon experiment ──────────────────────
# Edit SP_30C_VARIANTS / SP_30C_TRAJECTORIES to run a subset.
# Recommended order: '30M' first (fastest), then '30M_81', then '60M'.

SP_30C_SAVE_DIR    = os.path.join(HERE, 'runs', 'sp_30cages')
SP_30C_VARIANTS    = ['30M', '30M_81', '60M']   # edit to restrict
SP_30C_TRAJECTORIES = list(TRAJECTORY_BLOCKS)    # edit to restrict
SP_30C_PH_KWARGS   = {'K': 200, 'mip_gap': 0.03}
SLACK_THRESHOLD_KG = 100.0
os.makedirs(SP_30C_SAVE_DIR, exist_ok=True)

def run_sp_trajectory(variant, trajectory, units_df0, loc_mab, regional_mab,
                       temp_map, S_normal, S_bad, ph_kwargs, save_dir,
                       start_cal=0, slack_threshold=SLACK_THRESHOLD_KG):
    """Run one (variant × trajectory) SP rolling-horizon combination."""
    cfg    = VARIANT_CFG[variant]
    T      = cfg['T'];  commit = cfg['commit'];  n_stages = cfg['n_stages']
    slices = cfg['stage_slices'] or make_stage_slices(n_stages)
    macro  = cfg.get('macro_block_len', 1)
    blocks = TRAJECTORY_BLOCKS[trajectory]
    starts = list(range(0, REAL_HORIZON_SP, commit))

    out_dir  = os.path.join(save_dir, variant.lower(), trajectory)
    pkl_path = os.path.join(out_dir, 'result.pkl')
    if os.path.exists(pkl_path):
        existing = load_result(pkl_path)
        if not existing.get('config', {}).get('partial', True):
            print(f'  [SKIP] {variant} / {trajectory}')
            return existing
    os.makedirs(out_dir, exist_ok=True)

    max_months = REAL_HORIZON_SP + T
    real_temps, real_S = build_realized_sp(blocks, max_months, temp_map, S_normal, S_bad, start_cal)

    units_df = units_df0.copy()
    cal      = start_cal % 12
    solves, fwd_logs = [], []
    stage_offsets = [sl[0] for sl in slices]
    overall_t0 = time.time()

    for k, t_start in enumerate(starts):
        n_impl       = min(commit, REAL_HORIZON_SP - t_start)
        stage_labels = derive_stage_labels(blocks, t_start, n_stages, stage_offsets)
        print(f'  [{variant}/{trajectory}] solve {k+1}/{len(starts)} t={t_start} '
              f'labels={stage_labels}', flush=True)

        t_s0 = time.time()
        ald  = sp_rh.AugmentedLagrangianDecomposition(
            units_df=units_df, loc_mab=loc_mab, regional_mab=regional_mab,
            T=T, stage_slices=slices, start_calendar_month=cal,
            temps_normal=temp_map['normal'], temps_bad=temp_map['bad'],
            temps_good=temp_map['good'], S_normal=S_normal, S_bad=S_bad,
            macro_block_len=macro, **ph_kwargs,
        )
        ald.build();  ald.solve()
        wall = time.time() - t_s0

        feas = feasibility_weighted_expected_obj(ald, slack_threshold)
        try:
            s_idx   = find_scenario_idx_by_labels(ald, stage_labels);  matched='exact'
        except KeyError:
            s_idx   = find_best_match_scenario(ald, stage_labels);     matched='best'
        slack = scenario_slack_summary(ald, s_idx)
        try:
            decisions = extract_scenario_decisions(ald, s_idx, n_impl)
        except RuntimeError as e:
            print(f'    WARNING: {e}');  decisions = {'z':{}, 'q':{}, 'h_exist':{}, 'h':{}}

        temps_w = real_temps[t_start : t_start + n_impl]
        S_w     = real_S    [t_start : t_start + n_impl]
        new_df, fwd_log = advance_state_long(units_df, decisions, temps_w, S_w)
        for ev in fwd_log: ev['t_real_offset'] = t_start

        solves.append({
            'solve_idx': k, 't_start': t_start, 'horizon_months': T,
            'commit_months': commit, 'n_implement': n_impl,
            'stage_labels': list(stage_labels),
            'matched_scenario': ald.scenario_names[s_idx], 'matched_kind': matched,
            'ph_iters': int(getattr(ald,'n_iters',0)),
            'ph_eval_obj': float(getattr(ald,'eval_obj',0.0)),
            'wallclock_s': wall,
            'matched_scenario_slack_kg': slack,
            **{f'feas_{k_}': v for k_, v in feas.items()},
        })
        fwd_logs.append({
            'solve_idx': k, 't_start': t_start, 'n_implement': n_impl,
            'fwd_log': fwd_log, 'decisions': decisions,
            'realised_temps': list(map(float, temps_w)),
            'realised_S':     list(map(float, S_w)),
            'units_df_before': units_df.copy(), 'units_df_after': new_df.copy(),
        })
        units_df = new_df;  cal = (cal + n_impl) % 12

        # Incremental save
        _save_sp_result(out_dir, variant, trajectory, solves, fwd_logs,
                         units_df, cfg, ph_kwargs, start_cal, slack_threshold,
                         time.time()-overall_t0, partial=True)
        del ald

    result = _save_sp_result(out_dir, variant, trajectory, solves, fwd_logs,
                              units_df, cfg, ph_kwargs, start_cal, slack_threshold,
                              time.time()-overall_t0, partial=False)
    print(f'  Done {variant}/{trajectory}  total_wall={time.time()-overall_t0:.1f}s')
    return result

def _save_sp_result(out_dir, variant, trajectory, solves, fwd_logs,
                     units_df_final, cfg, ph_kw, start_cal, slack_thresh, wall, partial):
    result = {
        'config': {
            'variant': variant, 'trajectory': trajectory,
            'horizon_months': cfg['T'], 'commit_months': cfg['commit'],
            'n_stages': cfg['n_stages'], 'real_horizon_months': REAL_HORIZON_SP,
            'start_calendar_month': start_cal, 'ph_kwargs': ph_kw,
            'slack_threshold_kg': slack_thresh, 'total_wallclock_s': wall,
            'partial': partial, 'n_solves_completed': len(solves),
        },
        'solves': list(solves), 'real_state_log': list(fwd_logs),
        'units_df_final': units_df_final,
    }
    pkl = os.path.join(out_dir, 'result.pkl')
    tmp = pkl + '.tmp'
    with open(tmp, 'wb') as f: pickle.dump(result, f)
    os.replace(tmp, pkl)
    light = {'config': result['config'],
             'solves': [{k: v for k, v in s.items() if k != 'matched_scenario_slack_kg'}
                        for s in solves]}
    with open(os.path.join(out_dir, 'summary.json'), 'w') as f:
        json.dump(light, f, indent=2, default=str)
    return result

print(f'Running SP 30-cage: variants={SP_30C_VARIANTS}, trajectories={SP_30C_TRAJECTORIES}')
print(f'PH kwargs: {SP_30C_PH_KWARGS}')
t0_all = time.time()

for variant in SP_30C_VARIANTS:
    for traj in SP_30C_TRAJECTORIES:
        run_sp_trajectory(
            variant=variant, trajectory=traj,
            units_df0=units_df_30c, loc_mab=loc_mab_30c, regional_mab=regional_mab_30c,
            temp_map=temp_map_30c, S_normal=S_normal_30c, S_bad=S_bad_30c,
            ph_kwargs=SP_30C_PH_KWARGS, save_dir=SP_30C_SAVE_DIR,
        )

print(f'\nPart B complete. Total wall: {time.time()-t0_all:.1f}s')


In [ ]:
# ── Part B: SP 30-cage results summary ─────────────────────────────────────

rows_b = []
for variant in SP_30C_VARIANTS:
    for traj in SP_30C_TRAJECTORIES:
        pkl = os.path.join(SP_30C_SAVE_DIR, variant.lower(), traj, 'result.pkl')
        if not os.path.exists(pkl):
            print(f'  Missing: {variant}/{traj}')
            continue
        res  = load_result(pkl)
        npv  = compute_npv(res['real_state_log'], res['units_df_final'], REAL_HORIZON_SP)
        wall = res['config'].get('total_wallclock_s', 0)
        n_inf = sum(1 for s in res['solves'] if s.get('feas_n_feas', s.get('feas_n_total',1)) <
                    s.get('feas_n_total', 1))
        rows_b.append({
            'Variant': variant, 'Trajectory': traj,
            'NPV [MNOK]': round(npv / 1e6, 0),
            'Infeasible solves': n_inf,
            'Wall [h]': round(wall / 3600, 2),
        })

if rows_b:
    df_b = pd.DataFrame(rows_b)
    print('SP 30-cage results (NPV excludes feed costs):\n')
    display(df_b.pivot(index='Trajectory', columns='Variant', values='NPV [MNOK]'))

    print('\nFeasibility-weighted mean NPV per variant [MNOK]:')
    # Exclude trajectories with any infeasible solve
    feasible_df = df_b[df_b['Infeasible solves'] == 0]
    display(feasible_df.groupby('Variant')['NPV [MNOK]'].mean().to_frame())

    print('\nWall-clock time [hours]:')
    display(df_b.pivot(index='Trajectory', columns='Variant', values='Wall [h]'))


## Part C — Stochastic Horizon Comparison, 60-cage fleet with Deterministic Tail

Compares two SP variants over a **60-month** real horizon on the full 60-cage fleet.

| Variant | Horizon | Stochastic tree | Deterministic tail | Commit | Solves |
|---------|---------|-----------------|-------------------|--------|--------|
| `30M`   | 30 mo  | prefix(3) + 3×9mo stages → 27 scenarios | none | 15 mo | 4 |
| `60M`   | 60 mo  | prefix(3) + 3×9mo stages → 27 scenarios | 30-month all-normal tail | 30 mo | 2 |

Both variants commit half their planning horizon per roll. The 60M variant's
extra 30 months are a **deterministic continuation** (all-normal temperatures),
not additional stochastic branching. This tests whether a longer horizon —
even without more uncertainty representation — changes first-stage decisions.

Three in-sample paths: `normal`, `mixed`, `oscillating`.


In [ ]:
# ── Part C: SP 60-cage with deterministic tail ─────────────────────────────
# Uses sp_rh for the 30M variant and sp_60m for the 60M-with-det-tail variant.

REAL_HORIZON_60C = 60   # months

REALIZED_PATHS_60C = {
    'normal': {
        'regime_0': ('normal','normal','normal','normal'),
        'regime_1': ('normal','normal','normal','normal'),
    },
    'mixed': {
        'regime_0': ('normal','normal','normal','normal'),
        'regime_1': ('normal','good',  'normal','bad'),
    },
    'oscillating': {
        'regime_0': ('normal','good','normal','bad'),
        'regime_1': ('normal','good','normal','bad'),
    },
}

SP_60C_SAVE_DIR  = os.path.join(HERE, 'runs', 'sp_60cages')
SP_60C_PH_KWARGS = {'K': 200, 'mip_gap': 0.03}
os.makedirs(SP_60C_SAVE_DIR, exist_ok=True)

# Stage structure (30M and 60M share the same 27-scenario tree)
PREFIX_MONTHS_60C  = list(range(0, 3))          # 3-month deterministic prefix
STAGE_SLICES_60C   = [list(range(3, 12)),        # stage 1: 9 months
                       list(range(12, 21)),       # stage 2: 9 months
                       list(range(21, 30))]       # stage 3: 9 months  (30M)

def realized_monthly_labels_60c(path_spec):
    blocks = [
        (path_spec['regime_0'][0], 3), (path_spec['regime_0'][1], 9),
        (path_spec['regime_0'][2], 9), (path_spec['regime_0'][3], 9),
        (path_spec['regime_1'][0], 3), (path_spec['regime_1'][1], 9),
        (path_spec['regime_1'][2], 9), (path_spec['regime_1'][3], 9),
    ]
    return [lbl for lbl, n in blocks for _ in range(n)]

def realized_temps_S_60c(monthly_labels, start_cal, temp_map, S_normal, S_bad):
    n = len(monthly_labels)
    temps = np.array([temp_map[lbl][(start_cal + t) % 12] for t, lbl in enumerate(monthly_labels)])
    S     = np.full(n, S_normal)
    prev  = None
    for t, lbl in enumerate(monthly_labels):
        block_start = t if t == 0 or monthly_labels[t] != monthly_labels[t-1] else -1
        S[t] = S_bad if (lbl == 'good' and prev == 'good') else S_normal
        if t == 0 or lbl != monthly_labels[t-1]: prev = lbl
    return temps, S

def majority_label(labels, t_start, n_months):
    """Most common label in the real sequence for [t_start, t_start+n_months)."""
    window = labels[t_start:t_start+n_months]
    from collections import Counter
    return Counter(window).most_common(1)[0][0]

def run_sp_60c_variant(variant, path_name, path_spec, ph_kwargs, save_dir, start_cal=0):
    out_dir  = os.path.join(save_dir, variant, path_name)
    pkl_path = os.path.join(out_dir, 'result.pkl')
    if os.path.exists(pkl_path):
        existing = load_result(pkl_path)
        if not existing.get('config', {}).get('partial', True):
            print(f'  [SKIP] {variant}/{path_name}')
            return existing
    os.makedirs(out_dir, exist_ok=True)

    monthly = realized_monthly_labels_60c(path_spec)
    real_temps, real_S = realized_temps_S_60c(monthly, start_cal, temp_map_60c,
                                               S_normal_60c, S_bad_60c)
    if variant == '30M':
        horizon = 30;  commit = 15
        solve_starts = list(range(0, REAL_HORIZON_60C, commit))
    else:  # 60M
        horizon = 60;  commit = 30
        solve_starts = list(range(0, REAL_HORIZON_60C, commit))

    units_df = units_df_60c.copy()
    cal      = start_cal % 12
    solves, fwd_logs = [], []
    overall_t0 = time.time()

    for k, t_start in enumerate(solve_starts):
        n_impl = min(commit, REAL_HORIZON_60C - t_start)
        # Identify prefix label (majority of the first 3 months of the commit window)
        prefix_lbl = majority_label(monthly, t_start, 3)
        # Stage labels for the 27-scenario tree (stages 1-3, each 9 months)
        stage_labels = tuple(majority_label(monthly, t_start + 3 + 9*i, 9) for i in range(3))
        print(f'  [{variant}/{path_name}] solve {k+1}/{len(solve_starts)} '
              f't={t_start} prefix={prefix_lbl} stages={stage_labels}', flush=True)

        t_s0 = time.time()
        if variant == '30M':
            ald = sp_rh.AugmentedLagrangianDecomposition(
                units_df=units_df, loc_mab=loc_mab_60c, regional_mab=regional_mab_60c,
                T=horizon, stage_slices=STAGE_SLICES_60C,
                start_calendar_month=cal,
                temps_normal=temp_map_60c['normal'], temps_bad=temp_map_60c['bad'],
                temps_good=temp_map_60c['good'], S_normal=S_normal_60c, S_bad=S_bad_60c,
                prefix_months=PREFIX_MONTHS_60C, prefix_label=prefix_lbl,
                **ph_kwargs,
            )
        else:
            ald = sp_60m.AugmentedLagrangianDecomposition60M(
                units_df=units_df, loc_mab=loc_mab_60c, regional_mab=regional_mab_60c,
                start_calendar_month=cal,
                temps_normal=temp_map_60c['normal'], temps_bad=temp_map_60c['bad'],
                temps_good=temp_map_60c['good'], S_normal=S_normal_60c, S_bad=S_bad_60c,
                prefix_months=PREFIX_MONTHS_60C, prefix_label=prefix_lbl,
                **ph_kwargs,
            )
        ald.build();  ald.solve()
        wall = time.time() - t_s0

        # Find the matching scenario
        try:
            s_idx   = find_scenario_idx_by_labels(ald, stage_labels[:3]);  matched='exact'
        except (KeyError, AttributeError):
            s_idx   = find_best_match_scenario(ald, stage_labels[:3]);     matched='best'
        slack = scenario_slack_summary(ald, s_idx)
        try:
            decisions = extract_scenario_decisions(ald, s_idx, n_impl)
        except RuntimeError as e:
            print(f'    WARNING: {e}');  decisions = {'z':{}, 'q':{}, 'h_exist':{}, 'h':{}}

        feas  = feasibility_weighted_expected_obj(ald, SLACK_THRESHOLD_KG)
        temps_w = real_temps[t_start : t_start + n_impl]
        S_w     = real_S    [t_start : t_start + n_impl]
        new_df, fwd_log = advance_state_long(units_df, decisions, temps_w, S_w)
        for ev in fwd_log: ev['t_real_offset'] = t_start

        solves.append({
            'solve_idx': k, 't_start': t_start, 'horizon_months': horizon,
            'commit_months': commit, 'n_implement': n_impl,
            'prefix_label': prefix_lbl, 'stage_labels': list(stage_labels),
            'matched_scenario': ald.scenario_names[s_idx], 'matched_kind': matched,
            'ph_eval_obj': float(getattr(ald,'eval_obj',0.0)),
            'wallclock_s': wall, 'matched_scenario_slack_kg': slack,
            **{f'feas_{k_}': v for k_, v in feas.items()},
        })
        fwd_logs.append({
            'solve_idx': k, 't_start': t_start, 'n_implement': n_impl,
            'fwd_log': fwd_log, 'decisions': decisions,
            'realised_temps': list(map(float, temps_w)),
            'realised_S':     list(map(float, S_w)),
            'units_df_before': units_df.copy(), 'units_df_after': new_df.copy(),
        })
        units_df = new_df;  cal = (cal + n_impl) % 12

        # Incremental save
        _save_60c_result(out_dir, variant, path_name, solves, fwd_logs,
                          units_df, horizon, commit, ph_kwargs, start_cal,
                          time.time()-overall_t0, partial=True)
        del ald

    result = _save_60c_result(out_dir, variant, path_name, solves, fwd_logs,
                               units_df, horizon, commit, ph_kwargs, start_cal,
                               time.time()-overall_t0, partial=False)
    print(f'  Done {variant}/{path_name}  total_wall={time.time()-overall_t0:.1f}s')
    return result

def _save_60c_result(out_dir, variant, path_name, solves, fwd_logs,
                      units_df_final, horizon, commit, ph_kw, start_cal, wall, partial):
    result = {
        'config': {
            'variant': variant, 'path': path_name,
            'horizon_months': horizon, 'commit_months': commit,
            'real_horizon_months': REAL_HORIZON_60C,
            'ph_kwargs': ph_kw, 'total_wallclock_s': wall, 'partial': partial,
        },
        'solves': list(solves), 'real_state_log': list(fwd_logs),
        'units_df_final': units_df_final,
    }
    pkl = os.path.join(out_dir, 'result.pkl')
    tmp = pkl + '.tmp'
    with open(tmp, 'wb') as f: pickle.dump(result, f)
    os.replace(tmp, pkl)
    return result

# Run Part C
SP_60C_VARIANTS = ['30M', '60M']
print(f'Running SP 60-cage det-tail: variants={SP_60C_VARIANTS}')
t0_all = time.time()

for variant in SP_60C_VARIANTS:
    for path_name, path_spec in REALIZED_PATHS_60C.items():
        run_sp_60c_variant(variant, path_name, path_spec, SP_60C_PH_KWARGS, SP_60C_SAVE_DIR)

print(f'\nPart C complete. Total wall: {time.time()-t0_all:.1f}s')


In [ ]:
# ── Part C: SP 60-cage results summary ─────────────────────────────────────

rows_c = []
for variant in SP_60C_VARIANTS:
    for path_name in REALIZED_PATHS_60C:
        pkl = os.path.join(SP_60C_SAVE_DIR, variant, path_name, 'result.pkl')
        if not os.path.exists(pkl):
            print(f'  Missing: {variant}/{path_name}')
            continue
        res  = load_result(pkl)
        npv  = compute_npv(res['real_state_log'], res['units_df_final'], REAL_HORIZON_60C)
        wall = res['config'].get('total_wallclock_s', 0)
        rows_c.append({
            'Variant': variant, 'Path': path_name,
            'NPV [MNOK]': round(npv / 1e6, 0),
            'Wall [min]': round(wall / 60, 1),
        })

if rows_c:
    df_c = pd.DataFrame(rows_c)
    print('SP 60-cage results (NPV excludes feed costs):\n')
    display(df_c.pivot(index='Path', columns='Variant', values='NPV [MNOK]'))

    print('\nWall-clock time [minutes]:')
    display(df_c.pivot(index='Path', columns='Variant', values='Wall [min]'))

    print('\nObservation: does the 60M+det-tail variant consistently outperform 30M?')
    print(df_c.pivot(index='Path', columns='Variant', values='NPV [MNOK]').diff(axis=1).iloc[:, -1]
          .rename('60M - 30M [MNOK]').to_frame())
